# 학습한 LoRA 모델을 Colab에서 서빙하기 (vLLM + Cloudflare 터널)

Colab GPU에서 vLLM으로 **OpenAI 호환 API 서버**를 띄우고, 임시 공개 주소를 만들어
PC에서 돌아가는 sol_chat 웹 서버가 이 주소를 호출하게 합니다.
`chatbot/backends/api_backend.py` 를 그대로 쓰므로 코드 수정 없이 `configs/model.yaml` 만 바꾸면 됩니다.

- Colab 세션이 살아 있는 동안만 동작합니다.
- 실행할 때마다 주소와 키가 바뀝니다.

**시작 전**: `런타임 > 런타임 유형 변경 > T4 GPU`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q vllm

In [ ]:
import secrets
BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'   # configs/train.yaml 의 base_model 과 같게
ADAPTER = '/content/drive/MyDrive/sol_chat/outputs/adapters/sol-lora-v1'
LORA_NAME = 'sol-lora'                        # model.yaml 의 model 값으로 씀
LORA_RANK = 16                                # train.yaml 의 lora.r 이상
API_KEY = secrets.token_urlsafe(24)           # 이 서버를 아무나 못 쓰게 하는 키

In [ ]:
import subprocess
vllm = subprocess.Popen(
    ['vllm', 'serve', BASE_MODEL,
     '--enable-lora', '--lora-modules', f'{LORA_NAME}={ADAPTER}', '--max-lora-rank', str(LORA_RANK),
     '--api-key', API_KEY, '--port', '8000', '--max-model-len', '4096', '--dtype', 'half'],
    stdout=open('vllm.log', 'w'), stderr=subprocess.STDOUT)

In [ ]:
# 서버가 뜰 때까지 대기 (모델 다운로드 포함 수 분)
import time, requests
for _ in range(120):
    try:
        r = requests.get('http://localhost:8000/v1/models', headers={'Authorization': f'Bearer {API_KEY}'}, timeout=5)
        if r.ok:
            print('준비 완료:', [m['id'] for m in r.json()['data']])
            break
    except requests.ConnectionError:
        pass
    time.sleep(5)
else:
    print('서버가 뜨지 않았어요. 아래 로그를 확인하세요.')
    print(open('vllm.log').read()[-3000:])

In [ ]:
# Cloudflare 임시 터널로 공개 주소 만들기
import re
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                          stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
URL = None
for _ in range(30):
    time.sleep(2)
    found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if found:
        URL = found.group(0)
        break
print('공개 주소:', URL)

In [ ]:
print(f'''
# 1) PC의 .env 에 추가
COLAB_VLLM_API_KEY={API_KEY}

# 2) PC의 configs/model.yaml 의 models 아래에 추가 (기존 주석 항목을 이걸로 교체)
  my-lora-colab:
    label: "내 LoRA 모델 (Colab)"
    backend: api
    base_url: {URL}/v1
    model: {LORA_NAME}
    api_key_env: COLAB_VLLM_API_KEY
  qwen-base-colab:
    label: "학습 전 베이스 모델 (Colab)"
    backend: api
    base_url: {URL}/v1
    model: {BASE_MODEL}
    api_key_env: COLAB_VLLM_API_KEY

# 3) PC에서 서버 재시작 후 웹 화면의 모델 선택에서 고르기
''')

In [ ]:
# (확인) 여기서 바로 호출해 보기
from openai import OpenAI
client = OpenAI(base_url=f'{URL}/v1', api_key=API_KEY)
r = client.chat.completions.create(model=LORA_NAME, messages=[{'role': 'user', 'content': '안녕, 너는 누구야?'}], max_tokens=200)
print(r.choices[0].message.content)